# Вариант 24: RNN на NumPy

Все данные создаются локально. Сохранённые выводы получены настоящим запуском ячеек.

**Задача:** различить возрастающую последовательность и её обратную копию. Для этой задачи нейросеть не нужна: правило «последний элемент больше первого» тоже даёт 100%. Эксперимент нужен для изучения механики RNN.


In [1]:
from pathlib import Path
import sys, json, subprocess
import numpy as np
candidates = [Path.cwd(), Path.cwd() / "src", Path.cwd() / "rnn-24" / "src"]
SRC = next((p for p in candidates if (p / "run_demo.py").is_file()), None)
if SRC is None:
    raise RuntimeError("Откройте ноутбук из каталога темы или его папки src.")
sys.path.insert(0, str(SRC.resolve()))
import run_demo as demo
from numerical_example import calculate
print("NumPy:", np.__version__)


NumPy: 2.3.5


## 1. Скрытое состояние вручную

$h_t=\tanh(0.5x_t+0.8h_{t-1})$, $h_0=0$, входы $[1,0,-1]$. Веса заданы вручную; это отдельная иллюстрация, не веса обучаемой далее восьмимерной сети. На втором шаге вход равен нулю, но состояние сохраняет след первого шага.


In [2]:
for row in calculate()["steps"]:
    print(f"t={row['t']}: x={row['x']:+.0f}, input={row['input_term_0.5x']:+.10f}, "
          f"memory={row['memory_term_0.8h']:+.10f}, "
          f"sum={row['preactivation']:+.10f}, h={row['h']:+.10f}")


t=1: x=+1, input=+0.5000000000, memory=+0.0000000000, sum=+0.5000000000, h=+0.4621171573
t=2: x=+0, input=+0.0000000000, memory=+0.3696937258, sum=+0.3696937258, h=+0.3537237885
t=3: x=-1, input=-0.5000000000, memory=+0.2829790308, sum=-0.2170209692, h=-0.2136768637


## 2. Данные и контроль утечки

Длина 12. Каждая пара: возрастающая последовательность (метка 1) и обратная (метка 0). Train: 256 пар = 512 примеров. Test: 128 пар = 256 примеров. Разделение происходит **по ID пар**, стандартизация рассчитывается только по train.

Любая детерминированная функция только от среднего даст одинаковый ответ обоим членам пары и ошибётся ровно на одном: accuracy = 50%.


In [3]:
splits, scaling = demo.make_pairs()
train_ids = set(splits["train"]["pair_ids"].tolist())
test_ids = set(splits["test"]["pair_ids"].tolist())
assert train_ids.isdisjoint(test_ids)
print("Train:", splits["train"]["x"].shape, "Test:", splits["test"]["x"].shape)
print("Пересечение ID пар:", train_ids & test_ids)
baseline = demo.baselines(splits["test"])
for row in baseline["example_pair"]:
    print("label =", row["label"], "mean =", round(row["mean"], 10),
          "first/last =", np.round([row["values"][0], row["values"][-1]], 4))
print("Accuracy по среднему:", baseline["mean_threshold_accuracy"])
print("Accuracy по первому/последнему:", baseline["first_last_trend_accuracy"])


Train: (512, 12, 1) Test: (256, 12, 1)
Пересечение ID пар: set()
label = 0 mean = 1.9253915792 first/last = [2.3975 1.203 ]
label = 1 mean = 1.9253915792 first/last = [1.203  2.3975]
Accuracy по среднему: 0.5
Accuracy по первому/последнему: 1.0


## 3. Модель и проверка BPTT

$h_t=\tanh(x_tW_x+h_{t-1}W_h+b_h)$, $p=\sigma(h_TW_y+b_y)$. Векторы записаны строками, как в коде NumPy. Скрытых компонент 8, параметров 89. BCE считается через logits для численной устойчивости.

BPTT проходит через все 12 шагов. Сравним **все** аналитические производные с центральными конечными разностями на четырёх примерах. Проверка производится до clipping.


In [4]:
params = demo.init_params()
check, _ = demo.check_gradients(params, splits["train"]["x"][:4], splits["train"]["y"][:4])
print(json.dumps(check, ensure_ascii=False, indent=2))
assert check["passed"]


{
  "passed": true,
  "checked_parameters": 89,
  "epsilon": 1e-06,
  "max_absolute_error": 1.2305038238347166e-10,
  "max_relative_error": 4.2432599424133605e-08,
  "batch_size": 4,
  "sequence_length": 12,
  "stage": "initial parameters, analytical gradients before clipping"
}


## 4. Обучение

150 эпох, batch 64, Adam lr = 0.01, global gradient norm clipping до 1.0. Seeds: данные 24, модель 25, батчи 26. Test используется после обучения, без подбора по нему настроек. Ячейка запускает самостоятельный скрипт; интернет не нужен. Все результаты сохраняются в `results`.


In [5]:
run = subprocess.run([sys.executable, str(SRC / "run_demo.py")],
                     capture_output=True, text=True, check=True)
print(run.stdout.strip())


{
  "final": {
    "train": {
      "loss": 0.0009645370339605457,
      "accuracy": 1.0
    },
    "test": {
      "loss": 0.0014192691707632832,
      "accuracy": 1.0
    }
  },
  "gradient_check": {
    "passed": true,
    "checked_parameters": 89,
    "epsilon": 1e-06,
    "max_absolute_error": 1.2305038238347166e-10,
    "max_relative_error": 4.2432599424133605e-08,
    "batch_size": 4,
    "sequence_length": 12,
    "stage": "initial parameters, analytical gradients before clipping"
  },
  "optimization": {
    "updates": 1200,
    "clipped_updates": 25,
    "max_global_gradient_norm_before_clipping": 30.050531429922817
  },
  "output_dir": "<локальный каталог>/results"
}


## 5. Как читать результат

Сравним сеть с прозрачными правилами. 100% на ограниченной синтетике не означает 100% на произвольных рядах, речи или тексте. Один seed не даёт оценки устойчивости. Train loss может временно увеличиваться при обучении Adam; CSV сохраняет все реальные эпохи.


In [6]:
metrics = json.loads((SRC.parent / "results" / "metrics.json").read_text())
for split in ["train", "test"]:
    score = metrics["final"][split]
    base = metrics["baselines"][split]
    print(f"{split}: RNN accuracy={100*score['accuracy']:.1f}%, BCE={score['loss']:.10f}")
    print(f"       mean={100*base['mean_threshold_accuracy']:.1f}%, "
          f"first/last={100*base['first_last_trend_accuracy']:.1f}%")
print("История для графика: results/training_history.csv")


train: RNN accuracy=100.0%, BCE=0.0009645370
       mean=50.0%, first/last=100.0%
test: RNN accuracy=100.0%, BCE=0.0014192692
       mean=50.0%, first/last=100.0%
История для графика: results/training_history.csv


## Вопросы для обсуждения

1. Почему на втором шаге ручного примера состояние не равно нулю?
2. Почему среднее принципиально не различает направления пары?
3. Зачем держать прямую и обратную последовательности в одном split?
4. Почему правило first/last здесь не хуже RNN?
5. Что изменится, если класс зависит от первого элемента очень длинной последовательности?

Полная реализация: `run_demo.py`. Описание файлов и результатов: `README.md`.
